# Video Game Sales: Data Cleaning

Dataset: Video Game Sales with Ratings (Kaggle, 16,719 rows, 16 columns)

Goal: find the data problems, fix only what can be fixed honestly, and leave
the rest missing instead of inventing values.

In [2]:
import pandas as pd

raw = pd.read_csv("Video_Games_Sales_as_at_22_Dec_2016.csv")
print("Shape:", raw.shape)
raw.info()

Shape: (16719, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16717 non-null  object 
 1   Platform         16719 non-null  object 
 2   Year_of_Release  16450 non-null  float64
 3   Genre            16717 non-null  object 
 4   Publisher        16665 non-null  object 
 5   NA_Sales         16719 non-null  float64
 6   EU_Sales         16719 non-null  float64
 7   JP_Sales         16719 non-null  float64
 8   Other_Sales      16719 non-null  float64
 9   Global_Sales     16719 non-null  float64
 10  Critic_Score     8137 non-null   float64
 11  Critic_Count     8137 non-null   float64
 12  User_Score       10015 non-null  object 
 13  User_Count       7590 non-null   float64
 14  Developer        10096 non-null  object 
 15  Rating           9950 non-null   object 
dtypes: float64(9), object(7)
memory usage: 

In [3]:
print("Missing values per column:")
print(raw.isna().sum())

print("\nMost common User_Score values:")
print(raw["User_Score"].value_counts().head())

print("\nFully identical rows:", raw.duplicated().sum())
print("Same game, platform and year:",
      raw.duplicated(subset=["Name", "Platform", "Year_of_Release"]).sum())

Missing values per column:
Name                  2
Platform              0
Year_of_Release     269
Genre                 2
Publisher            54
NA_Sales              0
EU_Sales              0
JP_Sales              0
Other_Sales           0
Global_Sales          0
Critic_Score       8582
Critic_Count       8582
User_Score         6704
User_Count         9129
Developer          6623
Rating             6769
dtype: int64

Most common User_Score values:
User_Score
tbd    2425
7.8     324
8       290
8.2     282
8.3     254
Name: count, dtype: int64

Fully identical rows: 0
Same game, platform and year: 2


In [4]:
def clean_games(df):
    df = df.copy()
    df = df.dropna(subset=["Name", "Genre"])
    df = df.drop_duplicates(subset=["Name", "Platform", "Year_of_Release"])
    df["User_Score"] = pd.to_numeric(df["User_Score"], errors="coerce")
    df["Year_of_Release"] = df["Year_of_Release"].astype("Int64")
    return df

games = clean_games(raw)

print("Rows before:", len(raw))
print("Rows after: ", len(games))
print("Rows removed:", len(raw) - len(games))
print("'tbd' scores turned into missing:", (raw["User_Score"] == "tbd").sum())
print("User_Score type:", games["User_Score"].dtype)
print("Year_of_Release type:", games["Year_of_Release"].dtype)

Rows before: 16719
Rows after:  16716
Rows removed: 3
'tbd' scores turned into missing: 2425
User_Score type: float64
Year_of_Release type: Int64


In [5]:
assert games["Name"].notna().all(), "Rows with no name remain"
assert games["Genre"].notna().all(), "Rows with no genre remain"
assert not games.duplicated(subset=["Name", "Platform", "Year_of_Release"]).any(), "Duplicates remain"
assert games["User_Score"].dtype == "float64", "User_Score is not numeric"
print("All checks passed")

All checks passed


## Decisions

- Dropped rows with no Name or Genre: they cannot be used.
- Duplicates are the same game on the same platform in the same year, not just identical rows.
- "tbd" in User_Score means no score yet, so it became a missing value.
- Year_of_Release uses nullable Int64, so years stay whole numbers while gaps remain.
- Critic_Score, Critic_Count, User_Count, Developer and Rating are missing for about half the rows.
  They were left missing because filling them would invent reviews.